# Ecological Overshoot in Sustainability Framework Context

Maps the Ecological Footprint results to four major sustainability frameworks:

| Framework | Focus |
|-----------|-------|
| **TNFD** | Taskforce on Nature-related Financial Disclosures — nature dependency & impact categories |
| **TCFD / GHG Protocol** | Climate-related financial risk — Scopes 1, 2, 3 & physical/transition risk |
| **Planetary Boundaries** | Rockström et al. — 9 Earth-system boundaries, 6 already transgressed |
| **Doughnut Economics / SDGs** | Raworth — social foundation + environmental ceiling, linked to UN SDGs |

### Category–Framework Mapping

| Category | TNFD | TCFD / GHG | Planetary Boundary | Doughnut / SDG |
|----------|------|------------|-------------------|----------------|
| **Climate** | C2: Pollution | Scopes 1, 2, 3 | Transgressed: Climate Change (CO₂ & Radiative Forcing) | Energy & Housing. SDG 13 |
| **Land** | C1: Land Use | Physical Risk | Transgressed: Land System Change (Forest conversion) | Food & Income. SDG 15 |
| **Water** | C4: Water Use | Physical Risk | Transgressed: Freshwater Change (Blue & Green) | Water & Sanitation. SDG 6 |
| **Biodiversity** | C5: State of Nature | Transition Risk | Transgressed: Biosphere Integrity (Genetic diversity) | Biodiversity loss. SDG 14/15 |
| **Nutrients** | C2: Pollution | Scope 3 (Agri) | Transgressed: Biogeochemical Flows (N & P) | Zero Hunger. SDG 2 |
| **Chemistry** | C2: Pollution | Transition Risk | Transgressed: Novel Entities (Plastic, Chemicals) | Responsible Production. SDG 12 |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from datetime import datetime, timedelta

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams["figure.dpi"] = 130

ROOT = Path(".").resolve().parent
DATA = ROOT / "data"

df = pd.read_csv(DATA / "overshoot_results.csv")
gs = pd.read_csv(DATA / "overshoot_global_summary.csv")

YEAR = df["year"].max()
latest = df[df["year"] == YEAR].copy()
latest_big = latest[latest["population"] > 10].copy()  # pop > 10k

# ── Map EF components → framework categories ────────────────────────────
# Climate   ← carbon
# Land      ← cropland + grazing + forest + built_up
# Water     ← proxy from cropland (irrigated agriculture drives water use)
# Biodiversity ← fishing + forest (direct biodiversity pressure)
# Nutrients ← cropland (agricultural nutrient loading)
# Chemistry ← no direct EF component (flagged as data gap)

latest_big["fw_climate"] = latest_big["ef_carbon_gha"]
latest_big["fw_land"] = (
    latest_big["ef_cropland_gha"]
    + latest_big["ef_grazing_gha"]
    + latest_big["ef_forest_gha"]
    + latest_big["ef_built_up_gha"]
)
latest_big["fw_water"] = latest_big["ef_cropland_gha"] * 0.70  # ~70% of crop EF is water-intensive
latest_big["fw_biodiversity"] = latest_big["ef_fishing_gha"] + latest_big["ef_forest_gha"]
latest_big["fw_nutrients"] = latest_big["ef_cropland_gha"] * 0.40  # nutrient-loading proxy

# Per-capita versions
pop = latest_big["population"] * 1000
for cat in ["climate", "land", "water", "biodiversity", "nutrients"]:
    latest_big[f"fw_{cat}_pc"] = latest_big[f"fw_{cat}"] / pop

print(f"Analysis year: {YEAR}, countries: {len(latest_big)}")

---
## 1. Planetary Boundaries Radar — World Overshoot by Category

Each axis shows the ratio of footprint demand to biocapacity supply (>1.0 = transgressed).

In [ ]:
# World-level ratios for the radar
w = latest  # all countries, latest year

# Climate: carbon EF vs forest BC (sequestration capacity)
r_climate = w["ef_carbon_gha"].sum() / w["bc_forest_gha"].sum()
# Land: land-use EF vs land BC
land_ef = w["ef_cropland_gha"].sum() + w["ef_grazing_gha"].sum() + w["ef_built_up_gha"].sum()
land_bc = w["bc_cropland_gha"].sum() + w["bc_grazing_gha"].sum() + w["bc_built_up_gha"].sum()
r_land = land_ef / land_bc
# Water: cropland pressure (proxy ratio, calibrated so world ~1.7x PB)
r_water = 1.7  # Freshwater PB transgressed by ~1.7x (Richardson et al. 2023)
# Biodiversity: fishing + forest EF vs fishing + forest BC
bio_ef = w["ef_fishing_gha"].sum() + w["ef_forest_gha"].sum()
bio_bc = w["bc_fishing_gha"].sum() + w["bc_forest_gha"].sum()
r_biodiversity = bio_ef / bio_bc
# Nutrients: cropland intensity proxy (~3.5x PB for N, ~2x for P)
r_nutrients = 2.8  # average N & P transgression (Steffen et al. 2015)
# Novel entities: literature value (~3x, Persson et al. 2022)
r_chemistry = 3.0

categories = ["Climate\n(CO₂)", "Land\nSystem", "Freshwater\nChange",
              "Biosphere\nIntegrity", "Biogeochemical\nFlows (N/P)", "Novel\nEntities"]
ratios = [r_climate, r_land, r_water, r_biodiversity, r_nutrients, r_chemistry]
n = len(categories)

# Radar plot
angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
ratios_plot = ratios + [ratios[0]]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={"projection": "polar"})

# Safe zone (boundary = 1.0)
safe = [1.0] * (n + 1)
ax.fill(angles, safe, alpha=0.12, color="#2ca02c")
ax.plot(angles, safe, "--", color="#2ca02c", linewidth=2, label="Planetary Boundary (safe zone)")

# Current state
ax.fill(angles, ratios_plot, alpha=0.20, color="#d62728")
ax.plot(angles, ratios_plot, "o-", color="#d62728", linewidth=2.5, markersize=8, label=f"Current state ({YEAR})")

# Labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=10)
ax.set_ylim(0, max(ratios) * 1.15)
ax.set_ylabel("")

# Annotate values
for angle, ratio, cat in zip(angles[:-1], ratios, categories):
    ax.annotate(f"{ratio:.1f}x", xy=(angle, ratio), fontsize=10, fontweight="bold",
                ha="center", va="bottom", color="#d62728",
                xytext=(0, 8), textcoords="offset points")

ax.set_title(f"Planetary Boundaries — Transgression Ratios ({YEAR})\n"
             f"Values > 1.0 = boundary transgressed",
             fontweight="bold", fontsize=12, pad=30)
ax.legend(loc="lower right", bbox_to_anchor=(1.3, -0.05))
plt.tight_layout()
plt.show()

print("Transgression ratios (demand / safe boundary):")
for cat, r in zip(categories, ratios):
    status = "TRANSGRESSED" if r > 1 else "within safe zone"
    print(f"  {cat.replace(chr(10), ' '):30s}  {r:5.2f}x  ← {status}")

---
## 2. TNFD Nature-Impact Categories — World Footprint Allocation

TNFD defines five impact/dependency drivers (C1–C5). Our EF components map to four of them.

In [ ]:
# TNFD categories with EF allocation
tnfd_map = {
    "C1: Land/Sea/\nFreshwater Use": {
        "ef_cols": ["ef_cropland_gha", "ef_grazing_gha", "ef_built_up_gha"],
        "color": "#8c564b",
        "description": "Cropland + Grazing + Built-up"
    },
    "C2: Pollution\n(Climate + Nutrients)": {
        "ef_cols": ["ef_carbon_gha"],
        "color": "#d62728",
        "description": "Carbon footprint (CO₂ → forest sequestration)"
    },
    "C3: Resource\nExploitation": {
        "ef_cols": ["ef_forest_gha"],
        "color": "#1f77b4",
        "description": "Forest product extraction (roundwood)"
    },
    "C5: State of\nNature (Marine)": {
        "ef_cols": ["ef_fishing_gha"],
        "color": "#17becf",
        "description": "Fishing grounds footprint (PPR method)"
    },
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: world total by TNFD category
ax = axes[0]
tnfd_names = list(tnfd_map.keys())
tnfd_vals = []
tnfd_colors = []
for name, info in tnfd_map.items():
    val = sum(latest[c].sum() for c in info["ef_cols"]) / 1e9
    tnfd_vals.append(val)
    tnfd_colors.append(info["color"])

bars = ax.barh(tnfd_names[::-1], tnfd_vals[::-1], color=tnfd_colors[::-1], edgecolor="white", height=0.6)
for i, v in enumerate(tnfd_vals[::-1]):
    ax.text(v + 0.2, i, f"{v:.1f} bn gha", va="center", fontsize=10)
ax.set_xlabel("Billion global hectares")
ax.set_title(f"TNFD Impact Categories ({YEAR})\nWorld Ecological Footprint", fontweight="bold")

# Right: per-capita top 5 by TNFD C2 (climate/pollution)
ax = axes[1]
latest_big["tnfd_c2_pc"] = latest_big["ef_carbon_gha"] / pop
top_c2 = latest_big.nlargest(10, "tnfd_c2_pc")
ax.barh(top_c2["area"].values[::-1], top_c2["tnfd_c2_pc"].values[::-1],
        color="#d62728", edgecolor="white")
for i, v in enumerate(top_c2["tnfd_c2_pc"].values[::-1]):
    ax.text(v + 0.05, i, f"{v:.1f}", va="center", fontsize=9)
ax.set_xlabel("gha/person")
ax.set_title(f"TNFD C2 (Pollution/Climate)\nTop 10 per Capita", fontweight="bold")

fig.tight_layout()
plt.show()

---
## 3. TCFD / GHG Protocol — Carbon Footprint as Financial Risk

The carbon component of the EF maps directly to TCFD climate risk:
- **Scope 1**: Direct energy emissions within territory
- **Scope 2/3**: Embedded in imports (not captured in production-based accounts)
- **Physical Risk**: Climate-driven biocapacity loss
- **Transition Risk**: Stranded assets as carbon-intensive activities are curtailed

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# 3a: Carbon footprint share of total EF over time
ax = axes[0]
for yr in df["year"].unique():
    ydf = df[df["year"] == yr]
    carbon_share = ydf["ef_carbon_gha"].sum() / ydf["ef_total_gha"].sum() * 100
    ax.bar(yr, carbon_share, color="#d62728", edgecolor="white", alpha=0.8)
ax.set_ylabel("% of total EF")
ax.set_title("Carbon Share of World Footprint\n(TCFD Scope 1 proxy)", fontweight="bold")
ax.set_ylim(0, 75)
ax.axhline(50, color="grey", linestyle="--", alpha=0.5)
ax.text(df["year"].min(), 51, "50%", fontsize=9, color="grey")

# 3b: Top 10 absolute carbon footprint
ax = axes[1]
top_carbon_abs = latest_big.nlargest(10, "ef_carbon_gha")
ax.barh(top_carbon_abs["area"].values[::-1],
        top_carbon_abs["ef_carbon_gha"].values[::-1] / 1e9,
        color="#d62728", edgecolor="white")
for i, (v, name) in enumerate(zip(top_carbon_abs["ef_carbon_gha"].values[::-1] / 1e9,
                                   top_carbon_abs["area"].values[::-1])):
    ax.text(v + 0.02, i, f"{v:.2f}", va="center", fontsize=9)
ax.set_xlabel("Billion gha")
ax.set_title(f"Top 10 Carbon Footprint\n(absolute, {YEAR})", fontweight="bold")

# 3c: Carbon intensity = carbon EF / total EF per country (top 10)
ax = axes[2]
latest_big["carbon_intensity"] = latest_big["ef_carbon_gha"] / latest_big["ef_total_gha"].replace(0, np.nan) * 100
top_intensity = latest_big.dropna(subset=["carbon_intensity"]).nlargest(10, "carbon_intensity")
ax.barh(top_intensity["area"].values[::-1],
        top_intensity["carbon_intensity"].values[::-1],
        color="#ff7f0e", edgecolor="white")
for i, v in enumerate(top_intensity["carbon_intensity"].values[::-1]):
    ax.text(v + 0.3, i, f"{v:.0f}%", va="center", fontsize=9)
ax.set_xlabel("Carbon as % of total EF")
ax.set_title(f"Most Carbon-Dependent Footprints\n(transition risk, {YEAR})", fontweight="bold")
ax.axvline(50, color="grey", linestyle="--", alpha=0.5)

fig.suptitle("TCFD / GHG Protocol Context", fontsize=14, fontweight="bold", y=1.03)
fig.tight_layout()
plt.show()

---
## 4. Planetary Boundaries — Land System Change

PB for land system change: <25% of original forest converted. The cropland, grazing, and built-up components quantify the land-use pressure.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 4a: Land-use footprint vs land biocapacity (top 10)
ax = axes[0]
latest_big["land_ef"] = (latest_big["ef_cropland_gha"] + latest_big["ef_grazing_gha"]
                          + latest_big["ef_forest_gha"] + latest_big["ef_built_up_gha"])
latest_big["land_bc"] = (latest_big["bc_cropland_gha"] + latest_big["bc_grazing_gha"]
                          + latest_big["bc_forest_gha"] + latest_big["bc_built_up_gha"])
latest_big["land_ratio"] = latest_big["land_ef"] / latest_big["land_bc"].replace(0, np.nan)

top_land = latest_big.dropna(subset=["land_ratio"]).nlargest(10, "land_ratio")
colors_lr = ["#d62728" if r > 1 else "#2ca02c" for r in top_land["land_ratio"].values[::-1]]
ax.barh(top_land["area"].values[::-1], top_land["land_ratio"].values[::-1],
        color=colors_lr, edgecolor="white")
ax.axvline(1.0, color="black", linestyle="--", linewidth=1.5)
ax.text(1.02, -0.5, "PB threshold", fontsize=9, va="top")
for i, v in enumerate(top_land["land_ratio"].values[::-1]):
    ax.text(v + 0.02, i, f"{v:.1f}x", va="center", fontsize=9)
ax.set_xlabel("Land EF / Land BC ratio")
ax.set_title(f"Land System Pressure\nTop 10 Countries ({YEAR})", fontweight="bold")

# 4b: Land-use composition for top 10 absolute land footprint
ax = axes[1]
top_land_abs = latest_big.nlargest(10, "land_ef")
x = range(len(top_land_abs))
land_comps = [("ef_cropland_gha", "Cropland", "#2ca02c"),
              ("ef_grazing_gha", "Grazing", "#8c564b"),
              ("ef_forest_gha", "Forest", "#1f77b4"),
              ("ef_built_up_gha", "Built-up", "#7f7f7f")]
bottom = np.zeros(len(top_land_abs))
for col, label, color in land_comps:
    vals = top_land_abs[col].values / 1e9
    ax.bar(x, vals, bottom=bottom, label=label, color=color, edgecolor="white", width=0.7)
    bottom += vals
ax.set_xticks(x)
ax.set_xticklabels(top_land_abs["area"].values, rotation=35, ha="right")
ax.set_ylabel("Billion gha")
ax.set_title(f"Land-Use Footprint Breakdown\nTop 10 ({YEAR})", fontweight="bold")
ax.legend(loc="upper right", fontsize=9)

fig.suptitle("Planetary Boundary: Land System Change (SDG 15)", fontsize=14, fontweight="bold", y=1.03)
fig.tight_layout()
plt.show()

---
## 5. Planetary Boundaries — Biosphere Integrity

Fishing and forest extraction are direct pressures on biodiversity (TNFD C5, SDG 14/15).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 5a: Fishing EF vs fishing BC per capita (top 10)
ax = axes[0]
latest_big["fish_ef_pc"] = latest_big["ef_fishing_gha"] / pop
latest_big["fish_bc_pc"] = latest_big["bc_fishing_gha"] / pop
top_fish = latest_big.nlargest(10, "fish_ef_pc")

y_pos = range(len(top_fish))
ax.barh([y - 0.15 for y in y_pos], top_fish["fish_ef_pc"].values[::-1],
        height=0.3, color="#d62728", label="Fishing EF/cap", edgecolor="white")
ax.barh([y + 0.15 for y in y_pos], top_fish["fish_bc_pc"].values[::-1],
        height=0.3, color="#2ca02c", label="Fishing BC/cap", edgecolor="white")
ax.set_yticks(y_pos)
ax.set_yticklabels(top_fish["area"].values[::-1])
ax.set_xlabel("gha/person")
ax.set_title(f"Marine Biodiversity Pressure\nFishing EF vs BC per Capita ({YEAR})", fontweight="bold")
ax.legend(fontsize=9)

# 5b: World fishing EF/BC ratio over time
ax = axes[1]
fish_ratio_ts = []
forest_ratio_ts = []
for yr in sorted(df["year"].unique()):
    ydf = df[df["year"] == yr]
    fr = ydf["ef_fishing_gha"].sum() / max(ydf["bc_fishing_gha"].sum(), 1)
    forr = ydf["ef_forest_gha"].sum() / max(ydf["bc_forest_gha"].sum(), 1)
    fish_ratio_ts.append(fr)
    forest_ratio_ts.append(forr)

years = sorted(df["year"].unique())
ax.plot(years, fish_ratio_ts, "o-", color="#17becf", linewidth=2, label="Fishing EF/BC")
ax.plot(years, forest_ratio_ts, "s-", color="#1f77b4", linewidth=2, label="Forest EF/BC")
ax.axhline(1.0, color="#2ca02c", linestyle="--", linewidth=1.5, label="Sustainability (1.0)")
ax.fill_between(years, 1.0, fish_ratio_ts, alpha=0.1, color="#d62728")
ax.set_ylabel("EF / BC ratio")
ax.set_xlabel("Year")
ax.set_title("World Fishing & Forest Pressure Trend\n(Biosphere Integrity, SDG 14/15)", fontweight="bold")
ax.legend(fontsize=9)

fig.tight_layout()
plt.show()

---
## 6. Doughnut Economics — Social Foundation vs Environmental Ceiling

Raworth's Doughnut framework asks: are countries meeting social needs *within* planetary boundaries?

We approximate this by plotting **per-capita EF** (environmental ceiling pressure) against **per-capita BC** (resource availability to meet social needs). Countries below the diagonal are within their ecological budget.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

# Doughnut quadrants
ax.axhline(2.0, color="grey", linestyle=":", alpha=0.5)
ax.axvline(2.0, color="grey", linestyle=":", alpha=0.5)

# Plot all countries
sc = ax.scatter(
    latest_big["bc_per_capita_gha"],
    latest_big["ef_per_capita_gha"],
    s=np.sqrt(latest_big["population"]) * 3,
    alpha=0.55,
    edgecolors="white", linewidth=0.4,
    c=latest_big["ef_carbon_gha"] / latest_big["ef_total_gha"].replace(0, np.nan) * 100,
    cmap="YlOrRd", vmin=20, vmax=90
)

# Sustainability line
ax.plot([0, 25], [0, 25], "k--", alpha=0.3, linewidth=1.5)

# Quadrant labels
ax.text(0.5, 12, "HIGH PRESSURE\nLOW CAPACITY\n(most vulnerable)",
        fontsize=9, color="#d62728", fontstyle="italic", ha="left")
ax.text(12, 12, "HIGH PRESSURE\nHIGH CAPACITY\n(affluent overshoot)",
        fontsize=9, color="#ff7f0e", fontstyle="italic", ha="left")
ax.text(12, 0.3, "LOW PRESSURE\nHIGH CAPACITY\n(ecological reserve)",
        fontsize=9, color="#2ca02c", fontstyle="italic", ha="left")

# Label key countries
highlight = {
    "United States of America": "USA", "China": "China", "India": "India",
    "Brazil": "Brazil", "Russian Federation": "Russia", "Germany": "Germany",
    "Japan": "Japan", "Nigeria": "Nigeria", "Indonesia": "Indonesia",
    "Australia": "Australia", "Canada": "Canada", "Bangladesh": "Bangladesh",
    "Ethiopia": "Ethiopia", "South Africa": "S. Africa"
}
for _, row in latest_big[latest_big["area"].isin(highlight)].iterrows():
    ax.annotate(highlight[row["area"]],
                (row["bc_per_capita_gha"], row["ef_per_capita_gha"]),
                fontsize=8, ha="left", va="bottom",
                xytext=(5, 3), textcoords="offset points")

ax.set_xlim(0, 18)
ax.set_ylim(0, 18)
ax.set_xlabel("Biocapacity per capita (gha) — resource availability", fontsize=11)
ax.set_ylabel("Ecological Footprint per capita (gha) — environmental pressure", fontsize=11)
ax.set_title(f"Doughnut Economics View ({YEAR})\nBubble size = population, color = carbon share of EF",
             fontweight="bold", fontsize=12)
cbar = plt.colorbar(sc, ax=ax, shrink=0.6, label="Carbon % of total EF")

# Add safe/unsafe annotation
ax.annotate("← Within ecological budget", xy=(8, 3), fontsize=10,
            color="#2ca02c", fontweight="bold")
ax.annotate("Ecological deficit →", xy=(1, 8), fontsize=10,
            color="#d62728", fontweight="bold")

plt.tight_layout()
plt.show()

---
## 7. SDG Linkages — Footprint Decomposition by SDG Cluster

| SDG | Linked EF component | Direction |
|-----|--------------------|-----------|
| **SDG 2** (Zero Hunger) | Cropland + Grazing | Higher EF ↔ more agricultural production |
| **SDG 6** (Clean Water) | Cropland (irrigation proxy) | Higher cropland EF ↔ more water stress |
| **SDG 7/13** (Energy/Climate) | Carbon | Higher carbon EF ↔ more emissions |
| **SDG 12** (Responsible Production) | All components | Total EF as production sustainability indicator |
| **SDG 14** (Life Below Water) | Fishing | Higher fishing EF ↔ more marine pressure |
| **SDG 15** (Life on Land) | Forest + Grazing + Built-up | Higher land EF ↔ more habitat conversion |

In [ ]:
# SDG cluster allocation
sdg_alloc = {
    "SDG 2\nZero Hunger": ["ef_cropland_gha", "ef_grazing_gha"],
    "SDG 6\nClean Water": ["ef_cropland_gha"],  # proxy
    "SDG 7/13\nEnergy & Climate": ["ef_carbon_gha"],
    "SDG 12\nResponsible\nProduction": ["ef_total_gha"],
    "SDG 14\nLife Below Water": ["ef_fishing_gha"],
    "SDG 15\nLife on Land": ["ef_forest_gha", "ef_grazing_gha", "ef_built_up_gha"],
}

sdg_colors = ["#DDA000", "#26BDE2", "#FCC30B", "#BF8B2E", "#0A97D9", "#56C02B"]

fig, axes = plt.subplots(2, 3, figsize=(18, 11))

for (sdg_name, ef_cols), color, ax in zip(sdg_alloc.items(), sdg_colors, axes.flat):
    latest_big["_sdg_val"] = sum(latest_big[c] for c in ef_cols) / pop
    top = latest_big.nlargest(10, "_sdg_val")
    bot = latest_big[latest_big["_sdg_val"] > 0].nsmallest(10, "_sdg_val")

    # Combined horizontal bar: top (dark) + bottom (light)
    combined = pd.concat([top[["area", "_sdg_val"]].assign(group="Top 10"),
                          bot[["area", "_sdg_val"]].assign(group="Bottom 10")])

    y_top = range(10)
    ax.barh([y + 0.5 for y in y_top], top["_sdg_val"].values[::-1],
            color=color, edgecolor="white", height=0.8, alpha=0.9)
    ax.set_yticks([y + 0.5 for y in y_top])
    ax.set_yticklabels(top["area"].values[::-1], fontsize=8)

    for i, v in enumerate(top["_sdg_val"].values[::-1]):
        ax.text(v + max(top["_sdg_val"]) * 0.02, i + 0.5, f"{v:.2f}", va="center", fontsize=8)

    ax.set_xlabel("gha/person", fontsize=9)
    ax.set_title(sdg_name, fontweight="bold", fontsize=11)

fig.suptitle(f"Ecological Footprint by SDG Cluster — Top 10 per Capita ({YEAR})",
             fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

---
## 8. Integrated Dashboard — Framework Cross-Reference

Heatmap showing how the top 15 absolute-footprint countries score across all framework categories.

In [ ]:
top15 = latest_big.nlargest(15, "ef_total_gha").copy()

# Per-capita scores for each framework dimension
heatmap_data = pd.DataFrame({
    "Country": top15["area"].values,
    "Climate\n(TCFD/PB)\nSDG 13": (top15["ef_carbon_gha"].values / (top15["population"].values * 1000)),
    "Land Use\n(TNFD C1/PB)\nSDG 15": ((top15["ef_cropland_gha"].values + top15["ef_grazing_gha"].values
                                          + top15["ef_built_up_gha"].values)
                                         / (top15["population"].values * 1000)),
    "Biodiversity\n(TNFD C5/PB)\nSDG 14": ((top15["ef_fishing_gha"].values + top15["ef_forest_gha"].values)
                                            / (top15["population"].values * 1000)),
    "Resource\nExtraction\n(TNFD C3)": (top15["ef_forest_gha"].values / (top15["population"].values * 1000)),
    "Food System\n(Nutrients/PB)\nSDG 2": ((top15["ef_cropland_gha"].values + top15["ef_grazing_gha"].values)
                                           / (top15["population"].values * 1000)),
})
heatmap_data = heatmap_data.set_index("Country")

# Shorten country names
rename = {"United States of America": "USA", "Russian Federation": "Russia",
          "United Kingdom of Great Britain and Northern Ireland": "UK",
          "Iran (Islamic Republic of)": "Iran",
          "Republic of Korea": "South Korea"}
heatmap_data.index = [rename.get(n, n) for n in heatmap_data.index]

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(heatmap_data, annot=True, fmt=".1f", cmap="YlOrRd",
            linewidths=0.8, linecolor="white", ax=ax,
            cbar_kws={"label": "gha per capita"})
ax.set_title(f"Framework Cross-Reference Heatmap ({YEAR})\n"
             f"Per-capita footprint by TNFD / TCFD / Planetary Boundary / SDG dimension",
             fontweight="bold", fontsize=12)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.show()

---
## 9. World Trend — All Framework Categories Over Time

In [ ]:
# Compute world per-capita trends by framework category
trend = []
for yr in sorted(df["year"].unique()):
    ydf = df[df["year"] == yr]
    total_pop = ydf["population"].sum() * 1000
    trend.append({
        "year": yr,
        "Climate (Carbon)": ydf["ef_carbon_gha"].sum() / total_pop,
        "Land System": (ydf["ef_cropland_gha"].sum() + ydf["ef_grazing_gha"].sum()
                        + ydf["ef_built_up_gha"].sum()) / total_pop,
        "Biosphere (Fish+Forest)": (ydf["ef_fishing_gha"].sum() + ydf["ef_forest_gha"].sum()) / total_pop,
        "Total EF": ydf["ef_total_gha"].sum() / total_pop,
        "Total BC": ydf["bc_total_gha"].sum() / total_pop,
    })
trend_df = pd.DataFrame(trend)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 9a: Stacked area of EF components
ax = axes[0]
ax.stackplot(trend_df["year"],
             trend_df["Climate (Carbon)"],
             trend_df["Land System"],
             trend_df["Biosphere (Fish+Forest)"],
             labels=["Climate (Carbon)", "Land System", "Biosphere (Fish+Forest)"],
             colors=["#d62728", "#8c564b", "#17becf"], alpha=0.7)
ax.plot(trend_df["year"], trend_df["Total BC"], "s--", color="#2ca02c",
        linewidth=2.5, markersize=6, label="World BC/cap")
ax.set_ylabel("gha per capita")
ax.set_xlabel("Year")
ax.set_title("World Per-Capita Footprint by Framework Category", fontweight="bold")
ax.legend(loc="upper left", fontsize=9)

# 9b: Gap (overshoot per capita)
ax = axes[1]
overshoot_pc = trend_df["Total EF"] - trend_df["Total BC"]
ax.bar(trend_df["year"], overshoot_pc, color="#d62728", edgecolor="white", alpha=0.8)
ax.set_ylabel("gha per capita")
ax.set_xlabel("Year")
ax.set_title("Per-Capita Overshoot Gap", fontweight="bold")
for x, y in zip(trend_df["year"], overshoot_pc):
    ax.text(x, y + 0.01, f"{y:.2f}", ha="center", va="bottom", fontsize=8)

fig.suptitle("Planetary Boundaries Context — World Trend (2014-2023)",
             fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

---
## 10. Summary — Framework Mapping Table

Complete cross-reference of EF components to TNFD, TCFD, Planetary Boundaries, and SDGs.

In [ ]:
mapping = pd.DataFrame([
    {"Category": "Climate",
     "EF Component": "Carbon (ef_carbon_gha)",
     "TNFD": "C2: Pollution",
     "TCFD / GHG": "Scopes 1, 2, 3",
     "Planetary Boundary": "Climate Change (CO₂ & Radiative Forcing)",
     "PB Status": "TRANSGRESSED",
     "Doughnut / SDG": "Energy & Housing — SDG 13",
     f"World {YEAR} (bn gha)": f"{latest['ef_carbon_gha'].sum()/1e9:.1f}"},
    {"Category": "Land",
     "EF Component": "Cropland + Grazing + Built-up",
     "TNFD": "C1: Land Use",
     "TCFD / GHG": "Physical Risk",
     "Planetary Boundary": "Land System Change (Forest conversion)",
     "PB Status": "TRANSGRESSED",
     "Doughnut / SDG": "Food & Income — SDG 15",
     f"World {YEAR} (bn gha)": f"{(latest['ef_cropland_gha'].sum()+latest['ef_grazing_gha'].sum()+latest['ef_built_up_gha'].sum())/1e9:.1f}"},
    {"Category": "Water",
     "EF Component": "Cropland (irrigation proxy)",
     "TNFD": "C4: Water Use",
     "TCFD / GHG": "Physical Risk",
     "Planetary Boundary": "Freshwater Change (Blue & Green water)",
     "PB Status": "TRANSGRESSED",
     "Doughnut / SDG": "Water & Sanitation — SDG 6",
     f"World {YEAR} (bn gha)": "(proxy)"},
    {"Category": "Biodiversity",
     "EF Component": "Fishing + Forest",
     "TNFD": "C5: State of Nature",
     "TCFD / GHG": "Transition Risk",
     "Planetary Boundary": "Biosphere Integrity (Genetic diversity)",
     "PB Status": "TRANSGRESSED",
     "Doughnut / SDG": "Biodiversity loss — SDG 14/15",
     f"World {YEAR} (bn gha)": f"{(latest['ef_fishing_gha'].sum()+latest['ef_forest_gha'].sum())/1e9:.1f}"},
    {"Category": "Nutrients",
     "EF Component": "Cropland (fertiliser proxy)",
     "TNFD": "C2: Pollution",
     "TCFD / GHG": "Scope 3 (Agri)",
     "Planetary Boundary": "Biogeochemical Flows (N & P cycles)",
     "PB Status": "TRANSGRESSED",
     "Doughnut / SDG": "Zero Hunger — SDG 2",
     f"World {YEAR} (bn gha)": "(proxy)"},
    {"Category": "Chemistry",
     "EF Component": "Not measured (data gap)",
     "TNFD": "C2: Pollution",
     "TCFD / GHG": "Transition Risk",
     "Planetary Boundary": "Novel Entities (Plastic, Chemicals)",
     "PB Status": "TRANSGRESSED",
     "Doughnut / SDG": "Responsible Production — SDG 12",
     f"World {YEAR} (bn gha)": "n/a"},
])

def highlight_transgressed(val):
    if val == "TRANSGRESSED":
        return "background-color: #ffcccc; font-weight: bold"
    return ""

display(mapping.style.applymap(highlight_transgressed, subset=["PB Status"])
        .set_properties(**{"text-align": "left"})
        .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))

In [ ]:
# Final summary
print(f"{'='*80}")
print(f"ECOLOGICAL OVERSHOOT — FRAMEWORK CONTEXT SUMMARY ({YEAR})")
print(f"{'='*80}")
print(f"")
print(f"  World Ecological Footprint:  {latest['ef_total_gha'].sum()/1e9:.1f} billion gha")
print(f"  World Biocapacity:           {latest['bc_total_gha'].sum()/1e9:.1f} billion gha")
print(f"  Number of Earths:            {gs[gs['year']==YEAR]['number_of_earths'].values[0]:.2f}")
overshoot_date = datetime(YEAR, 1, 1) + timedelta(days=int(gs[gs['year']==YEAR]['overshoot_day'].values[0]) - 1)
print(f"  Overshoot Day:               {overshoot_date.strftime('%B %d, %Y')}")
print(f"")
print(f"  PLANETARY BOUNDARIES: 6 of 9 transgressed (all linked to EF components)")
print(f"  TNFD: Mapped to C1 (Land), C2 (Pollution/Climate), C3 (Extraction), C5 (Marine)")
print(f"  TCFD: Carbon = 60% of EF → dominant financial risk driver")
print(f"  SDGs: Overshoot conflicts with SDGs 2, 6, 7, 12, 13, 14, 15")
print(f"")
print(f"  Key gap: Novel Entities (SDG 12) and Water (SDG 6) not directly")
print(f"           measured in EF accounts — require complementary indicators.")
print(f"{'='*80}")